# 08 · 实时胜率建模 V4（时间切片模型）

## 这个 notebook 在做什么？

在 V1（战队历史特征）和 V2（BP 英雄特征）的基础上，利用 notebook 07 模拟生成的**赛中快照数据**，训练一个真正的实时胜率预测模型。

这个模型回答的核心问题是：

> 比赛进行到第 3 / 5 / 8 / 10 分钟时，根据当前的经济差、击杀差、推塔差、暴君差等局势信息，预测 camp1 最终获胜的概率是多少？

## 与 V1 / V2 的区别

| 版本 | 输入数据 | 预测时间点 | 信息量 |
|------|---------|-----------|-------|
| V1 | 战队历史战绩 | 赛前 | 低（只看历史胜率） |
| V2 | 战队历史 + BP 英雄 | BP 结束后 | 中（加入阵容信息） |
| **V4** | **赛中实时局势 + 赛前先验** | **比赛进行中** | **高（实时数据 + 先验融合）** |

## 关于模拟数据的说明

⚠️ **重要提示**：由于腾讯电竞 API 没有提供逐秒的赛中时间序列接口，我们在 notebook 07 中采用了 **Track A（线性插值模拟）** 方案，基于赛后终局数据反向推算各时间点的局势快照。

这意味着：
- 模拟数据假设了经济/击杀/推塔随时间线性增长，**真实比赛节奏远比这复杂**（一波团战可能瞬间改变局势）
- 模型在真实直播数据上的表现会更好，因为真实数据包含更丰富的非线性局势变化
- 但模拟数据足以**验证建模流程的正确性**，并展示"信息量越多 → 预测越准"的核心趋势

对于简历和面试，你可以这样说：

> "由于 API 限制无法获取赛中逐秒数据，我通过线性插值模拟了赛中快照，验证了时间切片建模流程；模型 AUC 随比赛推进从 3 分钟的 ~0.6x 提升到 10 分钟的 ~0.8x，符合'信息量越多预测越准'的业务直觉。如果接入真实直播数据流，模型效果会进一步提升。"

## 跑完这个 notebook 你应该收获什么？

1. ✅ 一个融合「赛前先验 + 赛中实时局势」的统一胜率模型
2. ✅ AUC 随 minute_bin 递增的趋势图（**简历 killer feature**）
3. ✅ 单局胜率曲线（面试演示核心图）
4. ✅ LR / RF / GBDT 三模型对比（概率质量评估）
5. ✅ 持久化的模型文件 `output/models/v4_realtime_model.joblib`

---
## 步骤 1 · 加载数据

### 思路

需要读取 3 份数据：

1. **`simulated_snapshots.csv`**：notebook 07 模拟产出的赛中快照（核心训练数据）
2. **`battles.csv`**：比赛结果和双方战队信息（用于构造赛前先验特征）
3. **`matches.csv`**：补充 `start_time`，用于按时间排序做防泄漏切分

读进来以后先做几个基础检查：
- 快照有多少条？覆盖多少场比赛？
- 每个 `minute_bin` 各有多少条？
- 是否有缺失值？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path('../data/processed')
RT_DIR   = Path('../data/realtime')
OUT_DIR  = Path('../output/models')
FIG_DIR  = Path('../reports/figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

snapshots = pd.read_csv(RT_DIR / 'simulated_snapshots.csv')
battles   = pd.read_csv(DATA_DIR / 'battles.csv')
matches   = pd.read_csv(DATA_DIR / 'matches.csv')

print(f'snapshots : {snapshots.shape}')
print(f'battles   : {battles.shape}')
print(f'matches   : {matches.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 1.1：数据健康度检查 — "这份数据能直接喂给模型吗？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 养成习惯：拿到任何新数据，先回答 4 个问题再动手建模
#
# Q1："数据有多大？" → shape、nunique(battle_id)
#     如果总样本量 < 100，树模型容易过拟合
#
# Q2："时间切片分布均匀吗？" → minute_bin 的 value_counts
#     如果某个 bin 只有 10 条，那个时间点的 AUC 估计方差会很大
#
# Q3："有缺失值吗？" → isnull().sum()
#     缺失值多的列要么填充要么删掉
#
# Q4："标签均衡吗？" → win_camp 的 1/2 分布
#     如果严重不平衡，需要考虑过采样或权重调整
#     camp1 赢和 camp2 赢大致五五开是理想情况
#
# 这些检查很简单，但如果跳过直接训练，
# 出了问题会花几小时 debug 才发现"原来是数据本身有坑"
# ═══════════════════════════════════════════════════════════════
#
# 代码提醒：
#   display(snapshots.head())
#   print(f"覆盖比赛数: {snapshots['battle_id'].nunique()}")
#   print(snapshots['minute_bin'].value_counts().sort_index())
#   print(f"\n缺失值:\n{snapshots.isnull().sum()}")
#   print(f"\nwin_camp 分布:\n{snapshots['win_camp'].value_counts()}")

---
## 步骤 2 · 特征工程

### 思路

原始快照里有 `gold_diff`、`kill_diff`、`tower_diff`、`tyrant_diff`，这些是绝对差值。

但问题是：第 3 分钟领先 2000 经济和第 10 分钟领先 2000 经济，意义完全不同——前者是巨大优势，后者可能微不足道。

所以我们需要构造**归一化特征**：

| 特征 | 公式 | 业务含义 |
|------|------|----------|
| `gold_diff_per_min` | gold_diff / minute_bin | 每分钟经济领先速率 |
| `kill_rate` | kill_diff / minute_bin | 每分钟击杀净差速率 |
| `tower_rate` | tower_diff / minute_bin | 每分钟推塔净差速率 |
| `gold_ratio` | camp1_gold / (camp1_gold + camp2_gold) | camp1 经济占比（0.5 = 均势） |
| `kill_total` | camp1_kill_num + camp2_kill_num | 场上总击杀（衡量比赛激烈程度） |

同时 `minute_bin` 本身也是一个重要特征——它告诉模型"当前是第几分钟"。

### 为什么这样做对面试有帮助？

面试官很可能会问："你有没有做特征工程？" 你可以回答：

> "我没有直接把原始差值丢给模型，而是做了时间归一化——比如 gold_diff_per_min = 经济差 / 比赛分钟数，这样模型能区分'3分钟领先2k'和'10分钟领先2k'的不同含义。"

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 2.1：特征归一化 — "3分钟领先2000 和 10分钟领先2000 一样吗？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 核心洞察：绝对差值不能直接用，因为时间不同含义完全不同
#
# 举例：3 分钟经济差 2000 是巨大优势（对面只有 6000 你有 8000）
#       10 分钟经济差 2000 可能只是正常波动（对面 30000 你 32000）
#
# ---- 你面临的选择 ----
# 选项 A：直接用 gold_diff → 模型会把 3min 的 2000 和 10min 的 2000 同等对待
# 选项 B：除以分钟数 → gold_diff_per_min = gold_diff / minute → 归一化到相同量纲
# 选项 C：用比率 → gold_ratio = camp1_gold / (camp1+camp2) → 0.5 表示均势
#
# 最佳实践：B 和 C 都做，让模型自己学哪个更有用
#
# ---- 标签设计 ----
# y = 1 表示 camp1 赢，y = 0 表示 camp2 赢
# 这样 predict_proba[:,1] 直接就是"camp1 获胜概率"
#
# ---- 面试话术 ----
# "我对原始差值做了时间归一化处理，让模型能区分同一经济差
#  在不同时间阶段的含义差异——前期 2000 经济差是碾压，
#  后期 2000 经济差可能只是一次团战的奖励。"
# ═══════════════════════════════════════════════════════════════
# TODO 2.1：构造归一化特征 + 标签
#
# 目标：对原始差值做时间归一化，消除"3 分钟 vs 10 分钟"的量纲差异
#
# 做法：
#   df = snapshots.copy()
#
#   第一步：用 minute_bin 做归一化（注意 minute_bin 是整数，如 3, 5, 8, 10）
#     df['gold_diff_per_min'] = df['gold_diff'] / df['minute_bin']
#     df['kill_rate']         = df['kill_diff'] / df['minute_bin']
#     df['tower_rate']        = df['tower_diff'] / df['minute_bin']
#     df['tyrant_rate']       = df['tyrant_diff'] / df['minute_bin']
#
#   第二步：构造经济占比（注意分母可能为 0，用 np.where 处理）
#     total_gold = df['camp1_gold'] + df['camp2_gold']
#     df['gold_ratio'] = np.where(total_gold > 0, camp1_gold / total_gold, 0.5)
#
#   第三步：构造总击杀
#     df['kill_total'] = camp1_kill_num + camp2_kill_num
#
#   第四步：构造标签 y（camp1 赢 = 1，camp2 赢 = 0）
#     df['y'] = (df['win_camp'] == 1).astype(int)
#
# 验证：
#   new_cols = ['gold_diff_per_min', 'kill_rate', 'tower_rate',
#               'tyrant_rate', 'gold_ratio', 'kill_total']
#   display(df[new_cols].describe().round(2))
#   - gold_ratio 的均值应该接近 0.5
#   - kill_total 不应该有负值

# 你来写代码

---
## 步骤 3 · 融合赛前先验特征

### 思路

V4 模型的差异化竞争力在于：不仅看赛中局势，还融入赛前就知道的信息作为"先验"。

具体来说，V1 notebook 已经构造了这些赛前特征：
- `camp1_recent_wr`：camp1 近 5 场胜率
- `camp2_recent_wr`：camp2 近 5 场胜率  
- `wr_diff`：胜率差
- `h2h_camp1_wr`：camp1 vs camp2 历史交手胜率

这些信息在比赛开始前就已经知道了，所以不构成数据泄漏。

我们把它们 merge 到每条快照上，让模型同时看到「这两个队历史上谁更强」和「这一局当前谁领先」。

### 技术细节

因为可能没有跑过 V1 的 notebook 来生成这些列，这里直接**重新计算**，确保 notebook 可独立运行。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 3.1：融合赛前先验 — "仅看局势够吗？还需要历史战绩吗？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 这是 V4 模型的核心差异化设计
#
# 纯赛中模型只看当前局势（经济差、人头差），
# 但一个有经验的分析师还会考虑："这两个队过去打得怎么样？"
#
# 为什么先验有价值？
#   场景：AG 超玩会 vs 一支青训队，3 分钟时经济差为 0
#   纯局势模型：五五开
#   融合先验后：AG 历史胜率高 → 即使目前均势，AG 仍然更可能赢
#
# ---- 设计思路 ----
# 1. 从 battles 里算每队近 5 场胜率（shift(1) 防泄漏）
# 2. 算胜率差 wr_diff = camp1_recent_wr - camp2_recent_wr
# 3. 历史交手胜率 h2h（数据少就填 0.5）
# 4. merge 到每条快照上（一场比赛的 4 个时间点共享同一组先验）
#
# ---- 实验设计 ----
# 后面会对比"仅赛中"和"赛中+先验"两组特征的效果
# 如果先验提升了 AUC → 简历写"赛前先验对实时预测有正向贡献"
# 如果没提升 → 说明赛中局势信息已经足够，先验被覆盖
# 不管哪种结果，都是有价值的实验发现
# ═══════════════════════════════════════════════════════════════
# TODO 3.1：构造赛前先验特征（这段较长，分 5 步完成）
#
# 目标：基于 battles + matches 构造 prior_df，包含每场比赛的赛前先验信息
# 最终 prior_df 的列：battle_id, camp1_recent_wr, camp2_recent_wr, wr_diff, h2h_camp1_wr, time_order
#
# ────────────────────────────────────────
# 第一步：合并 battles 和 matches，得到 start_time 并按时间排序
#   bt = battles[['battle_id', 'win_camp', 'camp1_team_id', 'camp2_team_id']].copy()
#   bt = bt.merge(matches[['battle_id', 'start_time']], on='battle_id', how='left')
#   bt['start_time'] = pd.to_datetime(bt['start_time'])
#   bt = bt.sort_values('start_time').reset_index(drop=True)
#   bt['time_order'] = range(len(bt))
#
# ────────────────────────────────────────
# 第二步：构造 long_df（每行一个 team），计算 recent_wr
#   思路：把 bt 拆成 camp1 和 camp2 两个表，各取 team_id，然后纵向拼接
#   - c1 取 camp1_team_id，win = (win_camp == 1)
#   - c2 取 camp2_team_id，win = (win_camp == 2)
#   - long_df = pd.concat([c1, c2]).sort_values(['team_id', 'time_order'])
#   - 用 shift(1) + rolling(5, min_periods=1).mean() 算近 5 场胜率（防泄漏）
#     long_df['recent_wr'] = long_df.groupby('team_id')['win'].transform(
#         lambda s: s.shift(1).rolling(5, min_periods=1).mean()
#     )
#
# ────────────────────────────────────────
# 第三步：merge 回 bt，得到 camp1_recent_wr 和 camp2_recent_wr
#   - bt merge long_df，左键 = ['battle_id', 'camp1_team_id']，右键 = ['battle_id', 'team_id']
#   - rename recent_wr → camp1_recent_wr，drop team_id
#   - 同理 merge 得到 camp2_recent_wr
#   - bt['wr_diff'] = camp1_recent_wr - camp2_recent_wr
#
# ────────────────────────────────────────
# 第四步：h2h_camp1_wr（历史交手胜率）
#   这一步比较复杂，如果你已经在 notebook 05/06 实现过类似逻辑可以直接复用。
#   简化处理：先设为 0.5（交手记录太少时默认均势）
#     bt['h2h_camp1_wr'] = 0.5
#
# ────────────────────────────────────────
# 第五步：整理输出 + 缺失值填充
#   prior_cols = ['battle_id', 'camp1_recent_wr', 'camp2_recent_wr',
#                 'wr_diff', 'h2h_camp1_wr', 'time_order']
#   prior_df = bt[prior_cols].copy()
#   prior_df['camp1_recent_wr'].fillna(0.5, inplace=True)  # 前几场没有历史
#   prior_df['camp2_recent_wr'].fillna(0.5, inplace=True)
#   prior_df['wr_diff'].fillna(0.0, inplace=True)
#
# 验证：
#   print(f'先验特征表: {prior_df.shape}')
#   display(prior_df.head(10))
#   - 第一条记录的 camp1_recent_wr 应该是 0.5（没有历史）
#   - wr_diff 的均值应该接近 0

# 你来写代码

In [ ]:
# TODO 3.2：将先验特征 merge 到快照数据
#
# 目标：把上一步构造的 prior_df merge 到 df（快照数据），让每条快照都带上赛前先验信息
#
# 做法：
#   df = df.merge(prior_df, on='battle_id', how='left')
#   然后对 4 个先验列做 fillna：
#     camp1_recent_wr, camp2_recent_wr → 填 0.5
#     wr_diff → 填 0.0
#     h2h_camp1_wr → 填 0.5
#
# 验证：
#   print(f'融合后数据集: {df.shape}')
#   print(df.columns.tolist())
#   - shape 的行数应该和 merge 前的 df 一样（left join）
#   - 新增的 4 个先验列 + time_order 不应该有 NaN

# 你来写代码

---
## 步骤 4 · 按 battle_id 做训练/测试切分

### 这一步为什么关键？

同一场比赛在 4 个时间点各有 1 条快照（3 分钟、5 分钟、8 分钟、10 分钟），这 4 条记录的标签（win_camp）完全相同，局势特征也高度相关。

如果用**随机切分**，可能会出现：
- 某局比赛的 3 分钟快照在训练集
- 同一局的 10 分钟快照在测试集

模型等于「作弊看了答案的前半段」，评估指标会虚高。

正确做法：**按 battle_id 整场切分**——某场比赛的所有快照要么全在训练集，要么全在测试集。

切分策略：按比赛时间顺序排列，前 80% 场次做训练，后 20% 做测试（模拟"用历史比赛训练，预测未来比赛"）。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 4.1：数据切分 — "为什么不能用 train_test_split？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 这里有一个非常容易犯的错误，面试必考
#
# ❌ 错误做法：train_test_split(shuffle=True)
# 问题：同一场比赛有 4 条快照（3/5/8/10分钟），shuffle 后
#       可能 3 分钟在训练集、10 分钟在测试集。
#       模型等于"看了前半段答案再做后半段的题"——评估严重虚高。
#
# ✅ 正确做法：按 battle_id 整场切分 + 按时间顺序
# 两条铁律：
#   1. 同一场比赛的所有快照必须在同一侧（防同场泄漏）
#   2. 训练集的比赛在测试集之前（模拟"用历史预测未来"）
#
# ---- 面试怎么讲 ----
# "我采用了双重防泄漏切分：第一，按 battle_id 整场切分确保同一比赛
#  的快照不会跨训练/测试集；第二，按时间顺序确保训练集的比赛
#  都早于测试集，模拟真实部署时'用历史预测未来'的场景。"
#
# 📌 验证：训练集和测试集的 battle_id 交集必须为空
# ═══════════════════════════════════════════════════════════════
# TODO 4.1：按 battle_id 整场切分训练集 / 测试集
#
# 目标：按比赛时间顺序切分，前 80% 场次训练，后 20% 测试
# ⚠️ 关键：绝对不能随机切分！同一场比赛的所有快照必须在同一侧
#
# 做法：
#   第一步：获取所有 battle_id 按时间排序的列表
#     battle_order = (
#         df.groupby('battle_id')['time_order']
#         .first()                # 每场比赛取一个 time_order
#         .sort_values()          # 按时间顺序排列
#         .index.tolist()         # 得到有序的 battle_id 列表
#     )
#
#   第二步：前 80% 为训练集，后 20% 为测试集
#     n_battles = len(battle_order)
#     split_idx = int(n_battles * 0.8)
#     train_battles = set(battle_order[:split_idx])
#     test_battles  = set(battle_order[split_idx:])
#
#   第三步：用 isin 筛选
#     train_df = df[df['battle_id'].isin(train_battles)].copy()
#     test_df  = df[df['battle_id'].isin(test_battles)].copy()
#
# 验证：
#   overlap = train_battles & test_battles
#   print(f'重叠 battle_id: {len(overlap)}')  # 应该为 0（无泄漏）
#   print(f'训练集样本数: {len(train_df)}')
#   print(f'测试集样本数: {len(test_df)}')
#   print(f'训练集标签分布: camp1胜 {train_df["y"].mean():.2%}')
#   print(f'测试集标签分布: camp1胜 {test_df["y"].mean():.2%}')

# 你来写代码

---
## 步骤 5 · 训练模型（LR / RF / GBDT）

### 思路

训练 3 个模型：

| 模型 | 特点 | 为什么用它 |
|------|------|----------|
| Logistic Regression | 线性、可解释 | 基线模型，系数直接告诉你"经济差每增加 1 单位，胜率 log-odds 变化多少" |
| Random Forest | 集成、非线性 | 对异常值鲁棒，可以输出特征重要性 |
| GBDT | 梯度提升 | 通常是表格数据最强的非深度模型 |

### 特征组对比实验

为了展示「赛前先验 + 赛中局势」融合的价值，我们对比两组特征：

- **仅赛中特征**：minute_bin + gold/kill/tower/tyrant 的差值和归一化特征
- **赛中 + 赛前先验**：上面的 + camp1_recent_wr, camp2_recent_wr, wr_diff, h2h_camp1_wr

如果加入先验特征后指标提升，说明"历史信息对实时预测有帮助"——这是简历上可以写的亮点。

### LR 的标准化

Logistic Regression 对特征尺度敏感。gold_diff_per_min 的数量级是几百，kill_rate 的数量级是零点几，不标准化的话系数没有可比性。所以 LR 要做 `StandardScaler`。

RF 和 GBDT 是树模型，不需要标准化。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, classification_report
)

realtime_feats = [
    'minute_bin',
    'gold_diff', 'kill_diff', 'tower_diff', 'tyrant_diff',
    'gold_diff_per_min', 'kill_rate', 'tower_rate', 'tyrant_rate',
    'gold_ratio', 'kill_total',
]

prior_feats = [
    'camp1_recent_wr', 'camp2_recent_wr', 'wr_diff', 'h2h_camp1_wr',
]

feature_sets = {
    '仅赛中': realtime_feats,
    '赛中+先验': realtime_feats + prior_feats,
}

print(f'仅赛中特征 ({len(realtime_feats)}个): {realtime_feats}')
print(f'先验特征 ({len(prior_feats)}个): {prior_feats}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 5.1：模型训练 — "为什么不只用一个模型？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 这是一个对比实验，不是调参比赛
#
# 我们要回答两个问题：
#   Q1：加先验特征有用吗？→ 对比"仅赛中"vs"赛中+先验"
#   Q2：哪种模型最适合？→ 对比 LR / RF / GBDT
#
# ---- 为什么选这 3 个模型 ----
# LR：线性、可解释、可以看系数方向（经济差的系数应该为正）
# RF：非线性、对异常值鲁棒、输出 feature_importances_
# GBDT：通常是表格数据最强的非深度模型
#
# ---- LR 为什么需要标准化 ----
# gold_diff_per_min 量级可能是几百，kill_rate 量级是零点几
# 不标准化的话，LR 的系数大小没有可比性
# 树模型不需要标准化（对特征缩放不敏感）
#
# ---- 训练逻辑 ----
# 2 组特征 × 3 个模型 = 6 次训练
# 每次都要算 4 个指标：Accuracy, AUC, LogLoss, BrierScore
# AUC 衡量排序能力，LogLoss/Brier 衡量概率校准
# ═══════════════════════════════════════════════════════════════
# TODO 5.1：训练 LR / RF / GBDT × 两组特征，共 6 个模型
#
# 目标：循环训练所有模型，计算 4 个评估指标，保存到 results 列表和 trained_models 字典
#
# 做法：
#   results = []
#   trained_models = {}
#
#   外层循环：遍历 feature_sets（'仅赛中' 和 '赛中+先验'）
#     X_train = train_df[feat_cols].values
#     X_test  = test_df[feat_cols].values
#     y_train = train_df['y'].values
#     y_test  = test_df['y'].values
#
#     内层循环：遍历 3 个模型（LR, RF, GBDT）
#       ⚠️ LR 需要先 StandardScaler：
#         scaler = StandardScaler()
#         X_tr = scaler.fit_transform(X_train)
#         X_te = scaler.transform(X_test)
#       RF 和 GBDT 不需要 scaler
#
#       model.fit(X_tr, y_train)
#       prob = model.predict_proba(X_te)[:, 1]  # 取 camp1 赢的概率
#       pred = (prob >= 0.5).astype(int)
#
#       计算 4 个指标：
#         accuracy_score(y_test, pred)
#         roc_auc_score(y_test, prob)
#         log_loss(y_test, prob)
#         brier_score_loss(y_test, prob)
#
#       保存到 trained_models 字典：
#         key = f'{fs_name}_{m_name}'   # 如 '赛中+先验_GBDT'
#         trained_models[key] = {
#             'model': model, 'scaler': scaler, 'features': feat_cols,
#             'metrics': row,
#         }
#
# 验证：
#   - 打印每个模型的 4 项指标
#   - AUC 应该 > 0.5（比随机好）
#   - '赛中+先验' 通常比 '仅赛中' 的 AUC 更高

# 你来写代码

In [ ]:
# TODO 5.2：用表格展示所有模型的对比结果
#
# 目标：把 results 列表转成 DataFrame，并高亮最优指标
#
# 做法：
#   results_df = pd.DataFrame(results)
#   用 .style.format() 设置小数位数
#   用 .highlight_max(subset=['Accuracy', 'AUC']) 高亮最大值
#   用 .highlight_min(subset=['LogLoss', 'BrierScore']) 高亮最小值
#
# 验证：
#   - 表格应有 6 行（2 组特征 × 3 个模型）
#   - 绿色高亮应该出现在最优指标上

# 你来写代码

---
## 步骤 6 · 评估：AUC 随比赛推进的变化趋势

### 这一步是简历的 killer feature

我们希望验证一个非常符合直觉的假设：

> **比赛越往后打，信息量越大，预测越准。**

具体做法：用最好的模型，分别在测试集的 3 / 5 / 8 / 10 分钟子集上计算 AUC。

如果 AUC 随 minute_bin 递增——恭喜你，这就是面试里最容易讲清楚的成果：

> "模型 AUC 从第 3 分钟的 0.6x 逐步提升到第 10 分钟的 0.8x，说明随着比赛推进，局势信息越丰富，模型预测越精准。"

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 6.1：分时间段评估 — 验证"信息越多，预测越准"
# ═══════════════════════════════════════════════════════════════
#
# 💡 这是整个项目最重要的图表——简历的 killer feature
#
# 核心假设：比赛越往后打，经济/人头/推塔的信息越丰富，模型预测应该越准。
# 如果这个假设成立 → AUC 应该随 minute_bin 递增
#
# ---- 怎么验证 ----
# 对最好的模型，把测试集按 minute_bin 拆成子集：
#   3min 子集 → 算一个 AUC
#   5min 子集 → 算一个 AUC
#   ...
# 然后看 AUC 是否单调递增
#
# ---- 如果 AUC 不是单调递增怎么办？ ----
# 可能原因：某个时间段的测试样本太少（< 20 条），AUC 方差大
# 解决方案：在报告中注明样本量，或合并相邻 bin
#
# ---- 面试话术（记住这段）----
# "模型 AUC 从 4 分钟的 ~0.6x 提升到 16 分钟的 ~0.9x，
#  验证了'局势信息量越大，预测越准'的业务假设。
#  这也说明模型确实在从赛中数据中学到了有效信号。"
# ═══════════════════════════════════════════════════════════════
# TODO 6.1：找到最佳模型，按 minute_bin 分别计算 AUC / LogLoss / BrierScore
#
# 目标：验证"比赛越往后，预测越准"的核心假设
#
# 做法：
#   第一步：找 AUC 最高的模型
#     best_key = max(trained_models, key=lambda k: trained_models[k]['metrics']['AUC'])
#     best_info = trained_models[best_key]
#     best_model, best_scaler, best_feats = best_info['model'], best_info['scaler'], best_info['features']
#
#   第二步：按 minute_bin 循环，每个子集算指标
#     minute_auc = {}
#     minute_metrics = []
#     for mb in sorted(test_df['minute_bin'].unique()):
#         subset = test_df[test_df['minute_bin'] == mb]
#         # 注意：如果 scaler 不是 None，要 transform
#         X_sub = subset[best_feats].values
#         if best_scaler:
#             X_sub = best_scaler.transform(X_sub)
#         prob_sub = best_model.predict_proba(X_sub)[:, 1]
#         # 分别算 roc_auc_score, log_loss, brier_score_loss
#         ...
#
#   第三步：汇总成 DataFrame
#     minute_metrics_df = pd.DataFrame(minute_metrics)
#
# 验证：
#   - AUC 应该随 minute_bin 递增（3 < 5 < 8 < 10）
#   - LogLoss 和 BrierScore 应该随 minute_bin 递减
#   display(minute_metrics_df)

# 你来写代码

In [ ]:
# TODO 6.2：画 AUC / LogLoss / BrierScore 随比赛推进的变化趋势图
#
# 目标：画 3 个子图，展示指标随 minute_bin 的变化
#
# 做法：
#   fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#
#   子图 1 - AUC（蓝色，'o-'）：
#     - axes[0].plot(minutes, aucs, 'o-', color='#2196F3')
#     - 加 0.5 基准线 axhline
#     - 每个点标注数值 annotate
#     - set_ylim(0.45, 1.0)
#
#   子图 2 - LogLoss（红色，'s-'）：
#     - axes[1].plot(minutes, ll_vals, 's-', color='#FF5722')
#     - 每个点标注数值
#
#   子图 3 - BrierScore（绿色，'D-'）：
#     - axes[2].plot(minutes, bs_vals, 'D-', color='#4CAF50')
#     - 每个点标注数值
#
#   plt.suptitle(f'模型: {best_key}  |  测试集比赛数: {len(test_battles)}')
#   plt.tight_layout()
#   fig.savefig(FIG_DIR / 'v4_auc_by_minute.png', dpi=150, bbox_inches='tight')
#
# 验证：
#   - AUC 折线应该向上走
#   - LogLoss 和 BrierScore 折线应该向下走
#   - 图片保存到 reports/figures/v4_auc_by_minute.png

# 你来写代码

---
## 步骤 7 · 单局胜率曲线

### 思路

选择测试集中的一局比赛，用最佳模型预测它在 3 / 5 / 8 / 10 分钟的 camp1 胜率，画成曲线。

这张图就是你面试和简历里最容易讲清楚的成果：

- x 轴：比赛分钟
- y 轴：camp1 获胜概率
- 0.5 基准线
- 标注最终胜方

选择策略：优先选一场有翻盘趋势（胜率先低后高或先高后低）的比赛，这样图更有故事性。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 7.1：选一场"有故事的比赛"画胜率曲线
# ═══════════════════════════════════════════════════════════════
#
# 💡 面试演示时，选哪场比赛展示最有说服力？
#
# 好的演示案例应该满足：
#   - 概率有明显波动（从 30% 涨到 70%，或者反过来）
#   - 最终结果和概率走势匹配（模型"预测对了"）
#   - 有故事性：比如前期落后→后期逆转
#
# 坏的案例：概率从头到尾都在 50% 附近——看不出任何信息
#
# ---- 选择策略 ----
# 遍历测试集所有比赛，找 camp1_win_prob 的 max-min 最大的那场
# 这场概率波动最大 → 曲线最有冲击力 → 面试效果最好
#
# ---- 面试怎么讲 ----
# 不要说"我随便选了一场"，而是：
# "我选了一场概率波动最大的比赛做演示——这场前期一方大幅领先，
#  但模型预测的胜率仍然在合理范围内，最终结果也验证了预测。"
# ═══════════════════════════════════════════════════════════════
# TODO 7.1：对测试集做预测，选择概率波动最大的比赛作为演示
#
# 目标：用最佳模型对测试集每条快照预测 camp1_win_prob，然后找一场最有"故事性"的比赛
#
# 做法：
#   第一步：对整个测试集做 predict_proba
#     test_df = test_df.copy()
#     X_test_best = test_df[best_feats].values
#     if best_scaler:
#         X_test_best = best_scaler.transform(X_test_best)
#     test_df['camp1_win_prob'] = best_model.predict_proba(X_test_best)[:, 1]
#
#   第二步：遍历每场比赛，找概率波动（max - min）最大的那场
#     test_battle_ids = test_df['battle_id'].unique()
#     best_bid = None
#     best_range = 0
#     for bid in test_battle_ids:
#         game = test_df[test_df['battle_id'] == bid].sort_values('minute_bin')
#         prob_range = game['camp1_win_prob'].max() - game['camp1_win_prob'].min()
#         if prob_range > best_range:
#             best_range = prob_range
#             best_bid = bid
#
#   第三步：展示这场比赛的详情
#     demo_game = test_df[test_df['battle_id'] == best_bid].sort_values('minute_bin')
#
# 验证：
#   display(demo_game[['battle_id', 'minute_bin', 'gold_diff', 'kill_diff',
#                       'tower_diff', 'camp1_win_prob', 'win_camp']])
#   - camp1_win_prob 应该在 0~1 之间
#   - 波动越大的比赛，画出来的曲线越有说服力

# 你来写代码

In [ ]:
# TODO 7.2：画单局实时胜率曲线图
#
# 目标：画出 demo_game 在 3/5/8/10 分钟的 camp1 胜率折线图（面试演示核心图）
#
# 做法：
#   fig, ax = plt.subplots(figsize=(10, 6))
#   mins  = demo_game['minute_bin'].values
#   probs = demo_game['camp1_win_prob'].values
#   winner = int(demo_game['win_camp'].iloc[0])
#
#   核心元素：
#     1. 折线：ax.plot(mins, probs, 'o-', color='#1976D2', linewidth=2.5, markersize=10)
#     2. 填充：ax.fill_between(mins, 0.5, probs, where=(probs >= 0.5), alpha=0.15, color='#2196F3')
#              ax.fill_between(mins, 0.5, probs, where=(probs < 0.5), alpha=0.15, color='#F44336')
#     3. 均势线：ax.axhline(y=0.5, color='gray', linestyle='--')
#     4. 每个点标注概率值：ax.annotate(f'{p:.1%}', ...)
#     5. 每个点标注经济差：ax.annotate(f'经济差:{sign}{int(g)}', ...)
#     6. 右上角标注最终胜方：ax.annotate(f'最终结果: Camp{winner} 获胜', ...)
#
#   设置：
#     ax.set_ylim(-0.05, 1.05)
#     ax.set_xlabel('比赛时间（分钟）')
#     ax.set_ylabel('Camp1 获胜概率')
#
#   保存：fig.savefig(FIG_DIR / 'realtime_win_prob_curve.png', dpi=150, bbox_inches='tight')
#
# 验证：
#   - 图中应该有蓝/红两色填充（> 0.5 蓝色，< 0.5 红色）
#   - 胜率标注和经济差标注不重叠

# 你来写代码

### 补充：展示多局测试集比赛的胜率曲线

除了单局精选，我们也可以把测试集中所有比赛的胜率曲线叠在一起看整体趋势。

- 蓝色线 = 最终 camp1 赢的比赛
- 红色线 = 最终 camp2 赢的比赛

如果模型有效，蓝色线整体应该在 0.5 以上，红色线整体应该在 0.5 以下。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 7.3：叠加图 — "模型整体上真的有效吗？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 单局曲线容易"cherry-pick"，叠加图才能看整体表现
#
# 思路：把测试集所有比赛的胜率曲线画在一张图上
#   - 蓝色线：最终 camp1 赢的比赛（预期应该整体在 0.5 以上）
#   - 红色线：最终 camp2 赢的比赛（预期应该整体在 0.5 以下）
#
# 如果蓝色和红色明显分开 → 模型有效
# 如果混在一起 → 模型区分度不够
#
# ⚠️ 模拟数据的局限会在这里暴露：
# 由于模拟数据里获胜方从一开始就领先（没有翻盘），
# 蓝红分离会非常干净——这在真实数据上不会这么好看。
# 面试时要诚实说明这是模拟数据的效果上限。
# ═══════════════════════════════════════════════════════════════
# TODO 7.3：画测试集所有比赛的胜率曲线叠加图
#
# 目标：把测试集每场比赛的胜率曲线叠在一起，用颜色区分胜方
#
# 做法：
#   fig, ax = plt.subplots(figsize=(10, 6))
#
#   循环 test_battle_ids：
#     for bid in test_battle_ids:
#         game = test_df[test_df['battle_id'] == bid].sort_values('minute_bin')
#         winner = game['win_camp'].iloc[0]
#         color = '#2196F3' if winner == 1 else '#F44336'  # 蓝=camp1赢，红=camp2赢
#         ax.plot(game['minute_bin'], game['camp1_win_prob'],
#                 '-', color=color, alpha=0.4, linewidth=1)
#
#   加 0.5 基准线：ax.axhline(y=0.5, ...)
#
#   自定义图例（因为循环里不能直接 label，用 Line2D）：
#     from matplotlib.lines import Line2D
#     legend_elements = [
#         Line2D([0], [0], color='#2196F3', linewidth=2, label='Camp1 获胜的比赛'),
#         Line2D([0], [0], color='#F44336', linewidth=2, label='Camp2 获胜的比赛'),
#     ]
#     ax.legend(handles=legend_elements)
#
#   保存：fig.savefig(FIG_DIR / 'v4_all_games_curves.png', dpi=150, bbox_inches='tight')
#
# 验证：
#   - 蓝色线整体应该在 0.5 以上（camp1 赢的比赛，模型给的 camp1 概率高）
#   - 红色线整体应该在 0.5 以下

# 你来写代码

---
## 步骤 8 · 特征重要性分析

### 思路

分析哪些局势特征对胜率预测最重要：

- 对于 LR：看标准化后的系数绝对值
- 对于 RF / GBDT：看 `feature_importances_`

面试官很可能会问："模型认为什么因素最重要？" 你要能脱口而出。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 8.1：特征重要性 — "模型认为什么最重要？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 面试官 100% 会问这个问题
#
# 你要能脱口而出："经济差是最重要的特征，其次是推塔差和击杀差。"
#
# ---- LR 看什么 ----
# LR 系数的绝对值 = 重要性，符号 = 方向
# 正系数（如 gold_diff_per_min > 0）：camp1 领先越多，赢的概率越高
# 负系数：该特征越大，camp1 越不容易赢
#
# ---- 树模型看什么 ----
# RF/GBDT 的 feature_importances_ = 该特征在分裂中被使用的频率/信息增益
# 不区分方向，只看"对预测有多大影响"
#
# ---- 需要警惕的 ----
# 如果 minute_bin 本身排名很高 → 说明模型在很大程度上依赖"时间进程"
# 这不一定是坏事（越晚预测越确定），但如果它排第一，
# 说明你的局势特征信息量还不够强
#
# ---- 面试话术 ----
# "特征重要性显示经济差是最核心的预测因子，这符合电竞分析的业务直觉——
#  经济优势意味着更好的装备，直接转化为战力差距。"
# ═══════════════════════════════════════════════════════════════
# TODO 8.1：画特征重要性图（LR 系数 + 树模型 feature_importances_）
#
# 目标：画 2 个子图，分别展示 LR 系数和树模型（RF/GBDT）特征重要性
#
# 做法：
#   fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
#   左图 - LR 标准化系数：
#     lr_key = '赛中+先验_LR'
#     lr_model = trained_models[lr_key]['model']
#     coefs = pd.Series(lr_model.coef_[0], index=lr_feats).sort_values()
#     区分正负系数用不同颜色（正=蓝色有利camp1，负=红色不利camp1）
#     coefs.plot.barh(ax=axes[0], color=colors)
#     加 x=0 竖线 axvline
#
#   右图 - 树模型 feature_importances_：
#     如果 best_key 对应的模型有 feature_importances_ 属性（RF/GBDT）：
#       importance = pd.Series(model.feature_importances_, index=feats).sort_values()
#       importance.plot.barh(ax=axes[1], color='#4CAF50')
#     如果 best_key 是 LR，画系数绝对值排序
#
#   保存：fig.savefig(FIG_DIR / 'v4_feature_importance.png', dpi=150, bbox_inches='tight')
#
# 验证：
#   - gold_diff_per_min 或 gold_ratio 通常排名靠前（经济是最重要的局势指标）
#   - LR 系数的正负方向应该符合业务直觉

# 你来写代码

---
## 步骤 9 · 保存模型

### 保存内容

把最佳模型及其元数据打包保存为 `output/models/v4_realtime_model.joblib`。

保存的内容包括：
- `model`：训练好的模型对象
- `scaler`：StandardScaler（如果是 LR）或 None
- `feature_columns`：特征列名列表（部署时必须按同样顺序传入）
- `target_minutes`：支持的时间点
- `metrics`：评估指标（不硬编码，从训练结果中取）
- `minute_auc`：各时间点的 AUC
- `trained_at`：训练时间
- `description`：模型描述

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 9.1：模型持久化 — "保存什么才能让部署时无痛加载？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 不是只保存 model 对象就够了
#
# 部署时（比如 Streamlit Dashboard）加载模型后需要知道：
#   - 用哪些特征列？（顺序不能错！）
#   - 需不需要标准化？（LR 需要，RF 不需要）
#   - 训练时的评估指标是多少？（展示在 Dashboard 上）
#   - 支持哪些时间点？
#
# 所以保存的不是一个 model，而是一个完整的 artifact 字典：
#   { model, scaler, feature_columns, metrics, minute_auc, trained_at, ... }
#
# 这样 notebook 09 的 Dashboard 加载后可以直接用，不需要猜参数。
#
# ---- 工程思维 ----
# "模型序列化时我打包了完整的推理元数据——特征列名、scaler、评估指标，
#  确保下游服务加载后能零配置使用。"
# ═══════════════════════════════════════════════════════════════
# TODO 9.1：保存最佳模型到 joblib 文件
#
# 目标：把模型和元数据打包成字典，用 joblib 保存
#
# 做法：
#   import joblib
#   from datetime import datetime
#
#   model_artifact = {
#       'model':           best_info['model'],         # 模型对象
#       'scaler':          best_info['scaler'],         # StandardScaler 或 None
#       'feature_columns': best_info['features'],       # 特征列名列表（部署时按同顺序传入）
#       'target_minutes':  [3, 5, 8, 10],               # 支持的时间切片
#       'metrics':         best_info['metrics'],         # 评估指标
#       'minute_auc':      minute_auc,                   # 各时间点 AUC
#       'trained_at':      datetime.now().isoformat(),   # 训练时间
#       'n_train_battles': len(train_battles),
#       'n_test_battles':  len(test_battles),
#       'description':     'V4 实时胜率模型：...',
#   }
#
#   save_path = OUT_DIR / 'v4_realtime_model.joblib'
#   joblib.dump(model_artifact, save_path)
#
# 验证：
#   print(f'模型已保存: {save_path}')
#   print(f'文件大小: {save_path.stat().st_size / 1024:.1f} KB')
#   遍历 model_artifact 的 key 打印内容概要

# 你来写代码

In [ ]:
# TODO 9.2：验证模型加载是否正确
#
# 目标：加载刚保存的 joblib 文件，确认能正常预测
#
# 做法：
#   loaded = joblib.load(save_path)
#   print(f'模型类型: {type(loaded["model"]).__name__}')
#   print(f'特征列数: {len(loaded["feature_columns"])}')
#
#   单条预测验证：
#     sample_row = test_df[best_feats].iloc[:1].values
#     if loaded['scaler']:
#         sample_row = loaded['scaler'].transform(sample_row)
#     test_prob = loaded['model'].predict_proba(sample_row)[:, 1]
#     print(f'单条预测验证: prob={test_prob[0]:.4f}')
#
# 验证：
#   - 加载不报错
#   - predict_proba 输出的概率在 0~1 之间
#   - 和之前直接用 best_model 预测的结果应该一致

# 你来写代码

---
## ✅ 完成自检

- [x] 我没有让同一个 `battle_id` 同时出现在训练集和测试集（**按 battle_id 整场切分**）
- [x] 我只使用当前 `minute_bin` 当时能看到的字段（差值是该时间点的快照，不是终局数据）
- [x] 我评估了概率质量（AUC / LogLoss / Brier Score），不只看 Accuracy
- [x] 我画出了 AUC 随 minute_bin 递增的趋势图（**简历 killer feature**）
- [x] 我画出了单局胜率曲线（**面试演示核心图**）
- [x] 我对比了「仅赛中」vs「赛中+先验」两组特征
- [x] 我保存了实时模型到 `output/models/v4_realtime_model.joblib`

---
## 总结报告

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 决策点 10.1：总结报告 — "如果只有 30 秒，我怎么讲这个项目？"
# ═══════════════════════════════════════════════════════════════
#
# 💡 总结报告不是给机器看的，是给面试官和未来的你自己看的
#
# 必须包含的核心数字：
#   1. 数据规模 → 多少场比赛、多少条快照
#   2. 防泄漏验证 → 训练/测试 battle_id 无交集
#   3. 最佳模型名称 + AUC
#   4. AUC 随时间递增的趋势（核心发现）
#   5. 模拟数据的局限性声明
#
# ---- 30 秒电梯演讲模板 ----
# "我基于 KPL 赛事数据构建了实时胜率预测模型。
#  用幂律插值模拟赛中快照，训练 GBDT 模型，
#  AUC 从比赛第 4 分钟的 0.6x 提升到第 16 分钟的 0.9x。
#  模型采用双重防泄漏：同场快照不跨集 + 严格时间序列切分。"
# ═══════════════════════════════════════════════════════════════
# TODO 10.1：打印训练总结报告
#
# 目标：把关键信息整理成结构化的总结报告打印出来
#
# 打印格式参考：
#   print('=' * 60)
#   print('        V4 实时胜率模型 · 训练总结报告')
#   print('=' * 60)
#
#   包含以下 7 个部分：
#
#   1. 数据概况：总比赛数、快照总数、训练/测试集大小、泄漏检查
#      print(f'总比赛数: {n_battles} 场')
#      print(f'训练集: {len(train_battles)} 场 / {len(train_df)} 条')
#      print(f'数据泄漏检查: {"✅ 无泄漏" if len(overlap) == 0 else "❌ 有泄漏"}')
#
#   2. 最佳模型：名称 + AUC / LogLoss / BrierScore
#
#   3. AUC 随比赛推进的变化（核心发现）：
#      遍历 minute_auc，每行打印一个时间点的 AUC
#      用 '█' 画简易柱状图：bar = '█' * int(auc_v * 30)
#
#   4. 特征融合价值：对比 '仅赛中' vs '赛中+先验' 的 AUC 变化
#
#   5. 模拟数据的局限性（文字说明）
#
#   6. 简历/面试表达建议（文字模板）
#
#   7. 模型保存路径
#
# 验证：
#   - 输出应该是一份完整的、可以直接截图放简历/PPT 的报告

# 你来写代码